# Assignment 2: ReAct Agent with a Search Tool

**Objective:** Build a simple ReAct (Reason + Act) agent that can decide when to use a search tool, inspect the tool result, and produce a grounded answer.

This notebook uses the Gemini API and a small local knowledge base. The search tool searches the supplied AI-related documents; it is not an internet search engine.

**How the loop works**
1. The user asks a question.
2. Gemini either responds directly or requests the `search_documents` tool.
3. Python runs the requested tool and sends its result back to Gemini.
4. Gemini uses the result to write the final answer.

> Run the notebook from top to bottom in Google Colab. Add your API key through Colab Secrets or enter it when prompted. Never paste an API key into a code cell or commit it to GitHub.

## 1. Install the Gemini SDK

We use Google's current `google-genai` Python SDK throughout this notebook. This avoids mixing the older `google-generativeai` package with the newer SDK.

In [3]:
%pip -q install -U google-genai

## 2. Configure your Gemini API key

Create a Gemini API key in [Google AI Studio](https://aistudio.google.com/app/apikey).

In Colab, you can add a secret named `GOOGLE_API_KEY` using the key icon in the left sidebar. If that secret is not available, the cell securely prompts for the key without displaying it. The key is kept only in this runtime.

In [4]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your Gemini API key: ")

if not api_key:
    raise ValueError("A Gemini API key is required to continue.")

os.environ["GOOGLE_API_KEY"] = api_key
print("Gemini API key loaded. It will not be printed.")

Gemini API key loaded. It will not be printed.


## 3. Create the Gemini client

The model name below uses Google's `gemini-flash-latest` alias. If that model is unavailable for your API account, replace it with a function-calling-capable Gemini model enabled for your account.

In [5]:
from google import genai
from google.genai import types

MODEL_NAME = "gemini-flash-latest"
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

print("Gemini client is ready.")

Gemini client is ready.


## 4. Prepare the knowledge base

The agent will search these short documents. Add more documents if you want to expand its knowledge.

In [6]:
DOCUMENTS = {
    "ai_basics": """
    Artificial intelligence (AI) is the broad field of creating computer systems
    that perform tasks that usually require human intelligence. Examples include
    understanding language, recognizing images, planning, and solving problems.
    """,
    "machine_learning": """
    Machine learning (ML) is a subset of artificial intelligence. It allows
    computer systems to learn patterns from data and improve their performance
    without being explicitly programmed with every rule. Common approaches
    include supervised learning, unsupervised learning, and reinforcement learning.
    """,
    "ai_applications": """
    AI is used in healthcare to support medical image analysis and clinical
    decision support. In education, AI can support personalized learning,
    tutoring systems, accessibility, and feedback. AI is also used in finance,
    transportation, agriculture, and customer service.
    """,
    "ai_ethics": """
    Important AI ethics concerns include fairness, privacy, transparency,
    accountability, safety, and human oversight. Systems should be evaluated
    for bias, personal data should be protected, and people should remain able
    to review important automated decisions.
    """
}

print(f"Knowledge base loaded: {len(DOCUMENTS)} documents.")

Knowledge base loaded: 4 documents.


## 5. Build the search tool

This is the action the agent can call. It scores each document by keyword overlap and returns the two most relevant matches. If nothing matches, it reports that no relevant document was found.

In [7]:
import re

def search_documents(query: str) -> str:
    """Search the local AI knowledge base for documents relevant to a query."""
    if not isinstance(query, str) or not query.strip():
        return "Please provide a non-empty search query."

    stop_words = {
        "what", "which", "when", "where", "why", "how", "the", "and",
        "for", "are", "is", "of", "in", "to", "a", "an", "with", "on",
        "does", "do", "can", "about", "explain", "tell", "me"
    }
    query_words = {
        word for word in re.findall(r"[a-zA-Z]+", query.lower())
        if word not in stop_words and len(word) > 2
    }

    if not query_words:
        return "No useful keywords were found in the query."

    scored_results = []
    for document_name, document_text in DOCUMENTS.items():
        document_words = set(re.findall(r"[a-zA-Z]+", document_text.lower()))
        score = len(query_words & document_words)
        if score:
            scored_results.append((score, document_name, document_text.strip()))

    scored_results.sort(key=lambda item: item[0], reverse=True)
    if not scored_results:
        return "No relevant documents found for that query."

    results = []
    for score, name, content in scored_results[:2]:
        results.append(f"Document: {name}\nRelevance score: {score}\nContent: {content}")
    return "\n\n".join(results)

# Quick test of the tool before connecting it to Gemini.
print(search_documents("What are the ethical concerns in AI?"))

Document: ai_ethics
Relevance score: 1
Content: Important AI ethics concerns include fairness, privacy, transparency,
    accountability, safety, and human oversight. Systems should be evaluated
    for bias, personal data should be protected, and people should remain able
    to review important automated decisions.


## 6. Declare the tool to Gemini

The function declaration tells Gemini the tool's name, purpose, and required input. We will execute the function ourselves so the ReAct steps are visible in the notebook.

In [8]:
search_declaration = types.FunctionDeclaration(
    name="search_documents",
    description="Search the local AI knowledge base and return the most relevant documents for a question.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The topic or question to search for."
            }
        },
        "required": ["query"]
    }
)

search_tool = types.Tool(function_declarations=[search_declaration])

SYSTEM_INSTRUCTION = """
You are a helpful study assistant. You have access to a search_documents tool
that searches a small local knowledge base about AI.

Use the tool when the question asks for factual information that may be in the
knowledge base. After receiving search results, answer using those results.
If the tool says no relevant documents were found, clearly say that the
knowledge base does not contain enough information. Do not invent facts or
pretend that the local search tool searched the internet. Keep answers clear
and suitable for a student.
"""

## 7. Implement the ReAct agent loop

The loop sends the user's question to Gemini. If Gemini requests a tool call, Python executes the search function and sends the result back. The loop stops when Gemini returns a normal text answer or the maximum number of tool rounds is reached.

In [9]:
import time

def generate_with_retry(**kwargs):
    """Retry temporary Gemini API overload/rate-limit errors with backoff."""
    for attempt in range(4):
        try:
            return client.models.generate_content(**kwargs)
        except Exception as exc:
            status = getattr(exc, "status_code", None)
            if status not in (429, 500, 502, 503, 504) or attempt == 3:
                raise
            wait_seconds = 2 ** attempt
            print(f"Gemini temporarily unavailable (HTTP {status}). Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

def run_agent(question: str, max_tool_rounds: int = 4) -> str:
    """Run a manual ReAct loop using Gemini function calling."""
    if not question or not question.strip():
        raise ValueError("Please enter a question.")

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=question.strip())]
        )
    ]

    for round_number in range(1, max_tool_rounds + 1):
        response = generate_with_retry(
            model=MODEL_NAME,
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                tools=[search_tool],
                temperature=0.2
            )
        )

        if not response.candidates:
            return "The model did not return a response. Please try again."

        model_content = response.candidates[0].content
        contents.append(model_content)

        function_calls = response.function_calls or []
        if not function_calls:
            return response.text or "The model returned no text answer."

        for function_call in function_calls:
            print(f"\n[Agent action] {function_call.name}({function_call.args})")

            if function_call.name == "search_documents":
                tool_result = search_documents(**function_call.args)
            else:
                tool_result = f"Unknown tool: {function_call.name}"

            print("[Tool result]")
            print(tool_result)

            contents.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_function_response(
                            name=function_call.name,
                            response={"result": tool_result}
                        )
                    ]
                )
            )

    # One final model call lets Gemini respond after the last tool result.
    final_response = generate_with_retry(
        model=MODEL_NAME,
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=[search_tool],
            temperature=0.2
        )
    )
    return final_response.text or "The agent reached its tool-call limit without a final text answer."

print("ReAct agent function is ready.")

ReAct agent function is ready.


## 8. Try the agent

Run the cell and observe when the model decides to use the search tool. Try your own questions after the provided examples.

In [14]:
questions = [
    "What is the difference between artificial intelligence and machine learning?",
    "What are the main applications of AI in healthcare and education?",
    "Why are fairness and privacy important in AI ethics?",
    "What are supervised and unsupervised learning?",
    "How many ghosts eat carrots every day?"
]

for index, question in enumerate(questions, start=1):
    print("=" * 80)
    print(f"Question {index}: {question}")
    print("-" * 80)
    try:
        answer = run_agent(question)
        print("\n[Final answer]")
        print(answer)
    except Exception as exc:
        print("\n[Question failed after retries]")
        print(f"{type(exc).__name__}: {exc}")
        print("Continuing to the next test question.")
    print()
    time.sleep(2)

Question 1: What is the difference between artificial intelligence and machine learning?
--------------------------------------------------------------------------------


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-flash-latest is not found for API version v1, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

## 9. Ask your own question

Change the question below and run the cell again.

In [13]:
my_question = "Explain why human oversight matters when using AI."
print(run_agent(my_question))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-flash-latest is not found for API version v1, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

## 10. Basic validation

This checks that the knowledge base, search function, and example question list are present. It does not replace running the Gemini-powered cells above.

In [12]:
assert len(DOCUMENTS) >= 4, "Add at least four knowledge-base documents."
assert callable(search_documents), "The search tool function is missing."
assert callable(run_agent), "The ReAct agent function is missing."
assert len(questions) >= 4, "Add at least four test questions."
assert "Add a question here." not in questions

print("Basic validation passed.")

Basic validation passed.


## Conclusion

The notebook demonstrates a ReAct-style agent: Gemini chooses whether to call a search tool, Python executes the tool, and Gemini uses the returned result to produce an answer.

**Before submission:** Run every cell from top to bottom in Google Colab, confirm that the example questions execute, and save the completed notebook. Do not include your API key in notebook cells, outputs, screenshots, or GitHub files.